In [ ]:
import time
import numpy as np
import pandas as pd
import openml
from pymfe.mfe import MFE

suite = openml.study.get_suite("OpenML-CC18")
tasks = openml.tasks.list_tasks(task_id=suite.tasks, output_format="dataframe")
print(len(tasks))
print(tasks.columns.tolist())

print(tasks.estimation_procedure.value_counts())

selected_tasks = tasks[
    tasks.NumberOfInstances.between(500, 50000) &
    (tasks.NumberOfFeatures <= 500)
]

print(len(selected_tasks))

#2) menor, mediana e maior base (instâncias x features)
tam = selected_tasks.assign(tam=selected_tasks.NumberOfInstances * selected_tasks.NumberOfFeatures).sort_values("tam")
amostra = tam.iloc[[0, len(tam) // 2, -1]]

#3) download do dataset com repetição se a conexão cair
def pegar_dados(did, alvo, tentativas=3):
    for k in range(tentativas):
        try:
            ds = openml.datasets.get_dataset(int(did))
            return ds.get_data(target=alvo, dataset_format="dataframe")
        except Exception as e:
            print("   falha ao baixar, tentando de novo:", repr(e)[:120])
            time.sleep(10)
    raise RuntimeError("não consegui baixar o dataset")

#4) pymfe: só esta parte entra na medida de tempo
def testar_pymfe(linha):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:                       #categorias -> números (faltante vira NaN)
        col = X.columns[i]
        cod = X[col].astype("category").cat.codes.astype(float)
        X[col] = cod.where(cod >= 0)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    X_arr = X.to_numpy(dtype=float)

    t0 = time.time()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X_arr, y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return len(nomes), time.time() - t0

#5) roda nas três bases
for linha in amostra.itertuples():
    print(linha.name, "|", linha.NumberOfInstances, "instâncias |", linha.NumberOfFeatures, "features")
    try:
        n, seg = testar_pymfe(linha)
        print("   ", n, "metacaracterísticas em", f"{seg:.1f}s")
    except Exception as e:
        print("    ERRO:", repr(e)[:300])

In [ ]:
import warnings

rng = np.random.default_rng(0)

def mf_com_codigos(linha, permutar):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        cod = X[col].astype("category").cat.codes.to_numpy()
        if permutar:
            perm = rng.permutation(cod.max() + 1)
            cod = np.where(cod >= 0, perm[np.clip(cod, 0, None)], -1)
        X[col] = np.where(cod >= 0, cod, np.nan).astype(float)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()

    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=float), y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

linha = next(selected_tasks[selected_tasks.name == "car"].itertuples())
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    a  = mf_com_codigos(linha, permutar=False)
    a2 = mf_com_codigos(linha, permutar=False)   # controle: mesma codificação, rodada de novo
    b  = mf_com_codigos(linha, permutar=True)

def mudou(x, z):
    return (x.isna() != z.isna()) | ((x - z).abs() > 1e-9)

print("controle (a vs a2):", mudou(a, a2).sum(), "de", len(a), "mudaram")
print("embaralhado (a vs b):", mudou(a, b).sum(), "de", len(a), "mudaram")
print(a[mudou(a, b)].index.tolist())

In [ ]:
def mf_freq(linha):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        freq = X[col].value_counts(normalize=True).to_dict()
        X[col] = X[col].astype(object).map(freq).astype(float)   # faltante continua NaN
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=float), y_arr, cat_cols=cat_cols)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    f = mf_freq(linha)

dif = mudou(a, f)
print(dif.sum(), "de", len(a), "diferem da codificação ordinal")
print(a[dif].index.tolist())

In [ ]:
print(selected_tasks.loc[selected_tasks.name == "car", "NumberOfInstances"].iloc[0], "=", 4*4*4*3*3*3)
X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
print(X.iloc[:, 0].value_counts())

In [ ]:
cand = selected_tasks[selected_tasks.name == "credit-g"]
print(len(cand), "base(s) encontrada(s)")
linha2 = next(cand.itertuples())

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    a_  = mf_com_codigos(linha2, permutar=False)
    a2_ = mf_com_codigos(linha2, permutar=False)
    b_  = mf_com_codigos(linha2, permutar=True)
    f_  = mf_freq(linha2)

print("controle      :", mudou(a_, a2_).sum(), "de", len(a_))
print("embaralhado   :", mudou(a_, b_).sum(), "de", len(a_))
print("frequência    :", mudou(a_, f_).sum(), "de", len(a_))
print(a_[mudou(a_, b_)].index.tolist())

In [ ]:
print(MFE.fit.__doc__)

In [ ]:
def mf_nativo(linha, transform_cat="gray", renomear=False):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i in cat_cols:
        col = X.columns[i]
        s = X[col].astype(object)
        if renomear:
            cats = list(s.dropna().unique())
            novo = dict(zip(cats, rng.permutation(len(cats))))
            s = s.map(lambda v: f"c{novo[v]}" if v in novo else v)
        X[col] = s
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat=transform_cat)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    g1 = mf_nativo(linha2, "gray")
    g2 = mf_nativo(linha2, "gray")                    # controle
    g3 = mf_nativo(linha2, "gray", renomear=True)     # nomes embaralhados
    oh = mf_nativo(linha2, "one-hot")

print("controle (gray vs gray)     :", mudou(g1, g2).sum(), "de", len(g1))
print("renomeado (gray vs gray')   :", mudou(g1, g3).sum(), "de", len(g1))
print("gray vs one-hot             :", mudou(g1, oh).sum(), "de", len(g1))
print("gray vs ordinal (a_)        :", mudou(g1, a_).sum(), "de", len(g1))
print(g1[mudou(g1, g3)].index.tolist())

In [ ]:
# 1) Quantas das 60 bases têm atributos categóricos (o alvo conta como simbólico, por isso o > 1)
com_cat = selected_tasks.NumberOfSymbolicFeatures > 1
print(com_cat.sum(), "de", len(selected_tasks), "bases têm atributos categóricos")
print(selected_tasks.loc[com_cat, ["name", "NumberOfSymbolicFeatures", "NumberOfNumericFeatures"]]
      .sort_values("NumberOfSymbolicFeatures", ascending=False).head(10))

# 2) Teste de invariância à ordem dos rótulos, por configuração
rng = np.random.default_rng(0)

def mudou(x, z):
    return (x.isna() != z.isna()) | ((x - z).abs() > 1e-9)

def resumo(x, z, rotulo):
    m = mudou(x, z)
    ok = m & x.notna() & z.notna()
    rel = ((x - z).abs() / x.abs().clip(lower=1e-12))[ok]
    print(f"   {rotulo:<10}: {int(m.sum()):>2} de {len(x)} mudaram | "
          f"mudança relativa mediana {rel.median():.1%} | acima de 10%: {int((rel > 0.10).sum())}")

for tc in ["gray", "one-hot-full", None]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        u1  = mf_nativo(linha2, tc)
        u1b = mf_nativo(linha2, tc)                  #controle: mesma configuração, de novo
        u2  = mf_nativo(linha2, tc, renomear=True)   #categorias renomeadas
    print(f"\ntransform_cat = {tc}")
    resumo(u1, u1b, "controle")
    resumo(u1, u2, "renomeado")
    print(f"   NaN: {int(u1.isna().sum())} | infinitos: {int(np.isinf(u1).sum())}")
    print("   instáveis no controle:", u1[mudou(u1, u1b)].index.tolist())

In [ ]:
for nome in ["kr-vs-kp", "splice", "PhishingWebsites", "credit-approval"]:
    lin = next(selected_tasks[selected_tasks.name == nome].itertuples())
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            t0 = time.time()
            u1 = mf_nativo(lin, "one-hot-full")
            seg = time.time() - t0
            u1b = mf_nativo(lin, "one-hot-full")
            u2 = mf_nativo(lin, "one-hot-full", renomear=True)
    except Exception as e:
        print(f"\n{nome}: ERRO {repr(e)[:200]}")
        continue
    print(f"\n{nome} ({seg:.1f}s)")
    resumo(u1, u1b, "controle")
    resumo(u1, u2, "renomeado")
    print(f"   NaN: {int(u1.isna().sum())} | infinitos: {int(np.isinf(u1).sum())}")
    print("   NaN em:", u1[u1.isna()].index.tolist())

In [ ]:
print((selected_tasks.NumberOfMissingValues > 0).sum(), "de", len(selected_tasks),
      "bases têm valores faltantes")

def mf_final(linha, transform_cat="one-hot-full"):
    X, y, cat_ind, _ = pegar_dados(linha.did, linha.target_feature)
    X = X.copy()
    cat_cols = [i for i, c in enumerate(cat_ind) if c]
    for i, col in enumerate(X.columns):
        if i in cat_cols:
            s = X[col].astype(object)
            moda = s.mode(dropna=True)
            X[col] = s.fillna(moda.iloc[0] if len(moda) else "faltante")
        else:
            s = pd.to_numeric(X[col], errors="coerce")
            X[col] = s.fillna(s.median() if s.notna().any() else 0.0)
    y_arr = pd.Series(y).astype("category").cat.codes.to_numpy()
    mfe = MFE(groups=["general", "statistical", "info-theory"])
    mfe.fit(X.to_numpy(dtype=object), y_arr, cat_cols=cat_cols, transform_cat=transform_cat)
    nomes, valores = mfe.extract()
    return pd.Series(valores, index=nomes, dtype=float)

for nome in ["credit-approval", "dna"]:
    lin = next(selected_tasks[selected_tasks.name == nome].itertuples())
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            t0 = time.time()
            u = mf_final(lin)
            seg = time.time() - t0
        print(f"\n{nome}: {seg:.1f}s | NaN: {int(u.isna().sum())} | infinitos: {int(np.isinf(u).sum())}")
        print("   NaN em:", u[u.isna()].index.tolist())
    except Exception as e:
        print(f"\n{nome}: ERRO {repr(e)[:200]}")